In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RAIN_FILE = "observed-timeseries-of-a__1_.csv"
FAO_FILE  = "FAOSTAT_data_en_9-28-2026.csv"

# --- Google Colab only: upload the files if they are not present ---
if not (os.path.exists(RAIN_FILE) and os.path.exists(FAO_FILE)):
    try:
        from google.colab import files
        print("Please upload the two CSV files")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put both CSV files in the same folder as this notebook.")

## 1. Load and inspect the data

In [ ]:
rain = pd.read_csv(RAIN_FILE, encoding="utf-8-sig")
fao  = pd.read_csv(FAO_FILE,  encoding="utf-8-sig")

print("Rainfall shape:", rain.shape)
display(rain.head())
print("\nFAOSTAT shape:", fao.shape)
display(fao.head())

Rainfall shape: (125, 3)


,Category,Precipitation (Annual Mean mm),5-year smooth (Gaussian Smooth)
0,1901,217.07,259.61
1,1902,350.09,278.29
2,1903,252.73,296.26
3,1904,351.37,312.85
4,1905,300.44,327.27



FAOSTAT shape: (192, 15)


,Domain Code,Domain,Area Code,Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,212,Syrian Arab Republic,5312,Area harvested,1717,"Cereals, primary",1961,1961,ha,2161829.0,A,Official value,NaN
1,QCL,Crops and livestock products,212,Syrian Arab Republic,5412,Yield,1717,"Cereals, primary",1961,1961,kg/ha,528.6,A,Official value,NaN
2,QCL,Crops and livestock products,212,Syrian Arab Republic,5510,Production,1717,"Cereals, primary",1961,1961,t,1142799.0,A,Official value,NaN
3,QCL,Crops and livestock products,212,Syrian Arab Republic,5312,Area harvested,1717,"Cereals, primary",1962,1962,ha,2268047.0,A,Official value,NaN
4,QCL,Crops and livestock products,212,Syrian Arab Republic,5412,Yield,1717,"Cereals, primary",1962,1962,kg/ha,987.1,A,Official value,NaN


In [ ]:
# Inspect: what does FAOSTAT contain? Any missing values / flags?
print(fao[["Element", "Item", "Unit"]].drop_duplicates().to_string(index=False))
print("\nYears:", fao["Year"].min(), "-", fao["Year"].max())
print("Data flags:", fao["Flag Description"].unique())
print("Missing values in Value column:", fao["Value"].isna().sum())
print("\nRainfall years:", rain["Category"].min(), "-", rain["Category"].max())
print("Missing values in rainfall:\n", rain.isna().sum().to_string())

       Element             Item  Unit
Area harvested Cereals, primary    ha
         Yield Cereals, primary kg/ha
    Production Cereals, primary     t

Years: 1961 - 2024
Data flags: <StringArray>
['Official value']
Length: 1, dtype: str
Missing values in Value column: 0

Rainfall years: 1901 - 2025
Missing values in rainfall:
 Category                           0
Precipitation (Annual Mean mm)     0
5-year smooth (Gaussian Smooth)    0


**Variables explained**
| Variable | Meaning | Unit |
|---|---|---|
| `rain_mm` | Annual mean precipitation over Syria | mm |
| `rain_smooth_5y` | 5-year Gaussian smooth of rainfall | mm |
| `area_ha` | Area harvested (cereals) | hectares |
| `yield_kg_ha` | Cereal yield | kg/ha |
| `production_t` | Cereal production | tonnes |

## 2. Preprocess

In [ ]:
# Rainfall: clean column names
rain = rain.rename(columns={
    "Category": "year",
    "Precipitation (Annual Mean mm)": "rain_mm",
    "5-year smooth (Gaussian Smooth)": "rain_smooth_5y"})
rain["year"] = rain["year"].astype(int)

# FAOSTAT: keep the useful columns, then pivot from long to wide (one row per year)
fao_small = fao[["Year", "Element", "Value"]].copy()
fao_small["Year"]  = fao_small["Year"].astype(int)
fao_small["Value"] = pd.to_numeric(fao_small["Value"], errors="coerce")

fao_wide = fao_small.pivot_table(index="Year", columns="Element", values="Value").reset_index()
fao_wide.columns.name = None
fao_wide = fao_wide.rename(columns={
    "Year": "year", "Area harvested": "area_ha",
    "Production": "production_t", "Yield": "yield_kg_ha"})
display(fao_wide.head())

,year,area_ha,production_t,yield_kg_ha
0,1961,2161829.0,1142799.0,528.6
1,1962,2268047.0,2238756.0,987.1
2,1963,2478681.0,2033301.0,820.3
3,1964,2349417.0,1791141.0,762.4
4,1965,2001153.0,1788800.0,893.9


## 3. Merge (inner join on year)
Only years present in both datasets are kept (1961–2024).

In [ ]:
df = rain.merge(fao_wide, on="year", how="inner").sort_values("year").reset_index(drop=True)

print("Merged rows:", len(df), "| years", df.year.min(), "-", df.year.max())
assert df["year"].is_unique
print("\nMissing values:\n", df.isna().sum().to_string())
display(df.head())

Merged rows: 64 | years 1961 - 2024

Missing values:
 year              0
rain_mm           0
rain_smooth_5y    0
area_ha           0
production_t      0
yield_kg_ha       0


,year,rain_mm,rain_smooth_5y,area_ha,production_t,yield_kg_ha
0,1961,319.08,269.26,2161829.0,1142799.0,528.6
1,1962,248.39,291.26,2268047.0,2238756.0,987.1
2,1963,410.53,313.13,2478681.0,2033301.0,820.3
3,1964,293.02,331.52,2349417.0,1791141.0,762.4
4,1965,332.71,345.25,2001153.0,1788800.0,893.9


## 4. Feature engineering
- **Anomaly:** rainfall minus its 1961–2024 mean
- **Detrended production/yield:** % deviation from a linear trend. This removes long-term growth from technology and inputs, so we can see the climate signal.
- **Drought flag:** rainfall below the 20th percentile
- **Period:** groups of years for comparison

In [ ]:
df["rain_prev_mm"]  = df["rain_mm"].shift(1)                    # last year's rainfall
df["rain_anom_mm"]  = df["rain_mm"] - df["rain_mm"].mean()
df["rain_anom_pct"] = 100 * df["rain_anom_mm"] / df["rain_mm"].mean()

def detrend_pct(series, years):
    slope, intercept, *_ = stats.linregress(years, series)
    trend = intercept + slope * years
    return 100 * (series - trend) / trend

df["production_t_detr_pct"] = detrend_pct(df["production_t"], df["year"])
df["yield_kg_ha_detr_pct"]  = detrend_pct(df["yield_kg_ha"],  df["year"])

DROUGHT_THRESHOLD = df["rain_mm"].quantile(0.20)
df["drought"] = df["rain_mm"] < DROUGHT_THRESHOLD
df["period"]  = pd.cut(df["year"], [1960, 1980, 2000, 2010, 2024],
                       labels=["1961-80", "1981-2000", "2001-10", "2011-24"])

print(f"Drought threshold (20th percentile): {DROUGHT_THRESHOLD:.1f} mm")
print("Drought years:", df.loc[df.drought, "year"].tolist())
df.to_csv("syria_rain_cereals_merged.csv", index=False)

Drought threshold (20th percentile): 260.2 mm
Drought years: [1962, 1970, 1973, 1989, 1990, 1995, 1999, 2005, 2007, 2008, 2010, 2017, 2021]
